# Regularization: keeping a model simple on purpose

Data Scientist track, lesson DS6. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** see regularization as a penalty on large coefficients, fit ridge and lasso models and compare their coefficients, scale the columns first, choose the penalty strength with cross-validation, and see where the penalty helps.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the app-events data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. The data is synthetic: a script made it from stated rules.

## Setup

In [ ]:
import math
import numbers
import os
import subprocess
import sys
import tempfile
import urllib.request

import numpy as np
import pandas as pd

try:
    import duckdb
except ImportError:  # Colab usually has DuckDB; install it when it does not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 2026       # the seed of the whole Data Scientist track

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The app-events tables (version 1), read from the release tag v0.2.0, or from the repository's own copy of the same files when the network is not available.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.2.0/datasets/app-events/v1/"
LOCAL = "../../datasets/app-events/v1/"      # the same files, when this notebook runs inside a clone of the repository


def fetch(name):
    """The path of one CSV file: downloaded from the tag, or the local copy."""
    try:
        path = os.path.join(tempfile.mkdtemp(), name + ".csv")
        urllib.request.urlretrieve(BASE + name + ".csv", path)
        return path
    except Exception:
        return LOCAL + name + ".csv"

con = duckdb.connect()                       # a new, empty database every time this cell runs
con.execute(f"CREATE TABLE users AS FROM read_csv('{fetch('users')}', types={{'signup_date': 'DATE'}})")
con.execute(f"CREATE TABLE events AS FROM read_csv('{fetch('events')}', types={{'event_ts': 'TIMESTAMP', 'received_at': 'TIMESTAMP'}})")

# Known answer from the dataset README: 21,841 people and 177,379 events.
assert con.execute("SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM events)").fetchall()[0] == (21841, 177379)

# The modeling table of the Data Scientist track: one row per person, built by the query of lesson DS3, "Building a modeling table in SQL".
# Run this cell as it is; DS3 explains it line by line.
MODELING_SQL = """WITH people AS (            -- one row per person: the spine of the table
  SELECT u.user_id, u.signup_date, u.channel, u.region, s.platform,
         s.event_ts AS signup_ts,
         s.event_ts + INTERVAL 7 DAY AS cutoff,       -- the prediction moment
         s.event_ts + INTERVAL 30 DAY AS horizon      -- the end of the label window
  FROM users AS u
  JOIN events AS s ON s.user_id = u.user_id AND s.event_name = 'signup_complete'
  WHERE u.signup_date BETWEEN DATE '2026-01-01' AND DATE '2026-05-31'
),
clean AS (                  -- retry copies removed: keep the first copy of each event
  SELECT *
  FROM events
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id, event_ts, event_name, platform ORDER BY received_at, event_id) = 1
),
known AS (                  -- what the app knew at the cutoff: it had happened and it had arrived
  SELECT p.user_id, c.event_name, c.event_ts, p.cutoff
  FROM people AS p
  JOIN clean AS c ON c.user_id = p.user_id
  WHERE c.event_ts <= p.cutoff AND c.received_at <= p.cutoff
),
features AS (
  SELECT user_id,
         COUNT(*) FILTER (WHERE event_name = 'session_start') AS sessions_7d,
         COUNT(*) FILTER (WHERE event_name = 'add_expense') AS expenses_7d,
         COUNT(*) FILTER (WHERE event_name IN ('session_start', 'add_expense') AND event_ts > cutoff - INTERVAL 3 DAY) AS recent_3d,
         COUNT(DISTINCT CAST(event_ts AS DATE)) FILTER (WHERE event_name IN ('session_start', 'add_expense')) AS active_days_7d,
         CAST(COUNT(*) FILTER (WHERE event_name = 'onboarding_complete') > 0 AS INTEGER) AS onboarded,
         CAST(COUNT(*) FILTER (WHERE event_name IN ('view_paywall', 'paywall_viewed')) > 0 AS INTEGER) AS saw_paywall,
         CAST(COUNT(*) FILTER (WHERE event_name = 'purchase') > 0 AS INTEGER) AS bought_plus
  FROM known
  GROUP BY user_id
),
label AS (                  -- from the other side of the cutoff: no visit and no expense in days 8 to 30
  SELECT p.user_id,
         CAST(NOT EXISTS (SELECT 1 FROM events AS e
                          WHERE e.user_id = p.user_id AND e.event_name IN ('session_start', 'add_expense')
                            AND e.event_ts > p.cutoff AND e.event_ts <= p.horizon) AS INTEGER) AS churned
  FROM people AS p
)
SELECT p.user_id, p.signup_date, p.channel, p.platform, p.region,
       f.sessions_7d, f.expenses_7d, f.recent_3d, f.active_days_7d, f.onboarded, f.saw_paywall, f.bought_plus,
       l.churned
FROM people AS p
JOIN features AS f ON f.user_id = p.user_id
JOIN label AS l ON l.user_id = p.user_id
ORDER BY p.user_id"""
table = con.sql(MODELING_SQL).df()
print(f"Data loaded: {len(table):,} people, {int(table['churned'].sum()):,} of them churned.")

## The split and the recipe

The columns, the split of lessons DS2 and DS4 (the test part stays untouched until the last unit) and the five folds of lesson DS5. `recipe` is the day-7 model with a penalty you can change: ridge or lasso, the number columns scaled or not. `auc_gain` is the gain with a bootstrap interval of lesson DS2. Run this cell as it is.

In [ ]:
STATIC = ["channel", "platform", "region"]                    # known at signup
WEEK1 = ["sessions_7d", "expenses_7d", "recent_3d", "active_days_7d", "onboarded", "saw_paywall", "bought_plus"]    # known at day 7
X, y = table[STATIC + WEEK1], table["churned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED, stratify=y)       # 13,362 and 4,454 people
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train)   # 10,021 and 3,341, cut from the training part
folds = StratifiedKFold(5, shuffle=True, random_state=SEED)       # the five folds of lesson DS5


def logit():
    """The day-7 model of lesson DS2: one-hot text columns, scaled number columns, logistic regression."""
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", StandardScaler(), WEEK1)]), LogisticRegression(max_iter=1000))


def auc_gain(p_new, p_base, y_true, resamples=2000):
    """ROC AUC of one model minus another on the same people, with a 95 percent bootstrap interval; both models are scored on the same resamples."""
    rng = np.random.default_rng(SEED)
    y_true, a, b = np.asarray(y_true), np.asarray(p_new), np.asarray(p_base)
    gains = []
    for _ in range(resamples):
        i = rng.integers(0, len(y_true), len(y_true))
        gains.append(roc_auc_score(y_true[i], a[i]) - roc_auc_score(y_true[i], b[i]))
    low, high = np.percentile(gains, [2.5, 97.5])
    return [round(float(roc_auc_score(y_true, a) - roc_auc_score(y_true, b)), 3) + 0.0, round(float(low), 3) + 0.0, round(float(high), 3) + 0.0]


def lasso_args(C):
    """The settings of an L1 penalty. scikit-learn 1.8 and later asks for l1_ratio=1; before 1.8 the same model is penalty="l1" (there l1_ratio=1 would be ignored)."""
    version = tuple(int(part) for part in sklearn.__version__.split(".")[:2])
    return dict(C=C, solver="saga", l1_ratio=1.0, random_state=SEED) if version >= (1, 8) else dict(C=C, solver="saga", penalty="l1", random_state=SEED)


def recipe(C=1.0, lasso=False, scaled=True, num=WEEK1):
    """The day-7 recipe with a penalty: ridge (L2, the default) or lasso (L1), the number columns scaled or left as they are."""
    settings = lasso_args(C) if lasso else dict(C=C)
    numbers = StandardScaler() if scaled else "passthrough"
    return make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), STATIC), ("num", numbers, num)]), LogisticRegression(max_iter=5000, **settings))


def kept(model):
    """The columns whose coefficient is not exactly zero."""
    names = [n.split("__")[1] for n in model[0].get_feature_names_out()]
    return [n for n, c in zip(names, model[-1].coef_[0]) if c != 0]


print(len(X_fit), len(X_val), len(X_test))

## Unit 1: A penalty on big coefficients

**Predict first.** As C falls from 1 to 0.001, the penalty grows a thousand times. Will the coefficients of the ridge reach exactly zero, or shrink and stay above it? And the lasso's?

In [ ]:
rows = []
for C in [1.0, 0.1, 0.03, 0.01, 0.003, 0.001]:
    ridge_coef = recipe(C).fit(X_fit, y_fit)[-1].coef_[0]                   # 19 coefficients: 12 text-column levels and 7 number columns
    lasso_coef = recipe(C, lasso=True).fit(X_fit, y_fit)[-1].coef_[0]
    rows.append((C, round(float(np.abs(ridge_coef).max()), 2), int((ridge_coef != 0).sum()), round(float(np.abs(lasso_coef).max()), 2), int((lasso_coef != 0).sum())))
print(pd.DataFrame(rows, columns=["C", "ridge: largest", "ridge: not zero", "lasso: largest", "lasso: not zero"]).to_string(index=False))

## Unit 2: Scale before you penalize

**Predict first.** Will the lasso keep the same columns on the raw number columns as on the scaled ones?

In [ ]:
kept_by = {}
for scaled in (True, False):
    kept_by[scaled] = kept(recipe(0.01, lasso=True, scaled=scaled).fit(X_fit, y_fit))
print("scaled:  ", kept_by[True])
print("unscaled:", kept_by[False])

## Unit 3: Choose C on the folds

**Predict first.** How much will the best C gain over C = 1 in the mean ROC AUC: about 0.0005, about 0.005 or about 0.02?

In [ ]:
grid = {"logisticregression__C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 100]}
searches = {}
for lasso in (False, True):
    searches[lasso] = GridSearchCV(recipe(lasso=lasso), grid, cv=folds, scoring="roc_auc").fit(X_train, y_train)     # the same five folds for every C
table_cv = pd.DataFrame({"C": grid["logisticregression__C"], "ridge": searches[False].cv_results_["mean_test_score"].round(4), "lasso": searches[True].cv_results_["mean_test_score"].round(4)})
print(table_cv.to_string(index=False))
for lasso in (False, True):
    best_row = int(searches[lasso].best_index_)
    scores = [searches[lasso].cv_results_[f"split{k}_test_score"][best_row] for k in range(5)]
    print("lasso" if lasso else "ridge", searches[lasso].best_params_["logisticregression__C"], round(float(min(scores)), 3), round(float(max(scores)), 3))

## Unit 4: Where the penalty pays: few people, many columns

**Predict first.** Which model will score best on the 3,341 other people, and how many of the 60 random columns will the lasso keep: none, a few, or about half?

In [ ]:
noise = pd.DataFrame(np.random.default_rng(SEED).normal(size=(len(table), 60)), index=table.index, columns=[f"noise{i}" for i in range(60)])   # 60 columns of random numbers
Xn = X.join(noise)
NUM = WEEK1 + list(noise.columns)
pilot = Xn.loc[X_fit.sample(500, random_state=SEED).index]            # a pilot of 500 people from the fitting part
y_pilot = y.loc[pilot.index]
grid_small = {"logisticregression__C": [0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]}
pilot_models = {"almost no penalty": recipe(1e6, num=NUM).fit(pilot, y_pilot),
                "ridge, C by CV": GridSearchCV(recipe(num=NUM), grid_small, cv=folds, scoring="roc_auc").fit(pilot, y_pilot).best_estimator_,
                "lasso, C by CV": GridSearchCV(recipe(lasso=True, num=NUM), grid_small, cv=folds, scoring="roc_auc").fit(pilot, y_pilot).best_estimator_}
for tag, m in pilot_models.items():
    chosen = kept(m)
    print(f"{tag:18s} AUC on 3,341 other people {roc_auc_score(y_val, m.predict_proba(Xn.loc[X_val.index])[:, 1]):.3f}   columns not zero {len(chosen):3d}   of them random {sum(c.startswith('noise') for c in chosen)}")

## Unit 5: The verdict on the test part

**Predict first.** Will either tuned model beat the day-7 model by more than 0.005 AUC?

In [ ]:
p_day7 = logit().fit(X_train, y_train).predict_proba(X_test)[:, 1]              # the test part is used once, here
tuned = {"ridge": searches[False].best_estimator_, "lasso": searches[True].best_estimator_}
p_tuned = {tag: m.predict_proba(X_test)[:, 1] for tag, m in tuned.items()}
print("day-7 model", round(float(roc_auc_score(y_test, p_day7)), 3), {tag: round(float(roc_auc_score(y_test, p)), 3) for tag, p in p_tuned.items()})
print({tag: auc_gain(p, p_day7, y_test) for tag, p in p_tuned.items()})

## Finish it

### Exercise 1: Finish it

**Task:** **Choose C on one validation part.** Fit the lasso at each C of `Cs` (given) on the fitting part, score each on the validation part with ROC AUC, and put the list [the C with the highest AUC, that AUC rounded to 3 decimals] in `answer_1`.

**Expected:** `answer_1` is a list of two numbers.

In [ ]:
Cs = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]       # given: the values of C to try
val_scores = []
for C in Cs:
    m = recipe(C, lasso=True).fit(X_fit, y_fit)
    # score this lasso on the validation part with ROC AUC and keep the number in val_scores
answer_1 = None  # replace None with [the C with the highest validation AUC, that AUC rounded to 3 decimals]

In [ ]:
_ = check("Exercise 1", "answer_1", [0.03, 0.7], traps=[([1.0, 0.692], "That scores on the fitting part, where the weakest penalty always looks best. Score on the validation part.")], tol=5e-4)

<details>
<summary>Hint</summary>

`roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])` is the score of one fitted lasso; `np.argmax` finds the position of the highest one in the list.

</details>

<details>
<summary>Half-solution</summary>

```python
val_scores.append(float(roc_auc_score(y_val, m.predict_proba(...)[:, 1])))
answer_1 = [Cs[int(np.argmax(val_scores))], round(max(val_scores), 3)]
```

</details>

<details>
<summary>Solution</summary>

```python
Cs = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
val_scores = []
for C in Cs:
    m = recipe(C, lasso=True).fit(X_fit, y_fit)
    val_scores.append(float(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])))
answer_1 = [Cs[int(np.argmax(val_scores))], round(max(val_scores), 3)]
```

The best C is 0.03, with 0.700 on the validation part. The scores from C = 0.03 to 1 are 0.6998, 0.6996, 0.6996, 0.6995: a flat top, so one validation part picks a point inside a region.

</details>

## On your own

### Exercise 2: On your own

**Task:** **Fewer people than columns.** Before you start, say in one sentence what the result looks like: two AUCs between 0.55 and 0.7, the second higher. Repeat unit 4 with a pilot of 250 people (`X_fit.sample(250, random_state=SEED)`; `Xn`, `NUM` and `grid_small` come from unit 4), and put [AUC with almost no penalty, AUC of the ridge with C chosen on `folds`] on the validation people in `answer_2`, each rounded to 3 decimals.

**Expected:** `answer_2` is a list of two numbers.

In [ ]:
_ = check("Exercise 2", "answer_2", [0.585, 0.636], traps=[([0.636, 0.585], "That is the list the wrong way round: the model with almost no penalty comes first.")], tol=5e-4)

<details>
<summary>Hint</summary>

Cut the pilot with `Xn.loc[X_fit.sample(250, random_state=SEED).index]`, fit `recipe(1e6, num=NUM)` on it, and put `recipe(num=NUM)` into `GridSearchCV(..., grid_small, cv=folds, scoring="roc_auc")`. Score on `Xn.loc[X_val.index]`.

</details>

<details>
<summary>Half-solution</summary>

Expected output in words: *"Two scores, the ridge clearly higher than the model with almost no penalty."*

```python
small = Xn.loc[X_fit.sample(250, random_state=SEED).index]
y_small = y.loc[small.index]
ridge_small = GridSearchCV(recipe(num=NUM), grid_small, cv=folds, scoring="roc_auc").fit(small, y_small).best_estimator_
```

</details>

<details>
<summary>Solution</summary>

```python
small = Xn.loc[X_fit.sample(250, random_state=SEED).index]
y_small = y.loc[small.index]
plain_small = recipe(1e6, num=NUM).fit(small, y_small)
ridge_small = GridSearchCV(recipe(num=NUM), grid_small, cv=folds, scoring="roc_auc").fit(small, y_small).best_estimator_
Xv_small = Xn.loc[X_val.index]
answer_2 = [round(float(roc_auc_score(y_val, m.predict_proba(Xv_small)[:, 1])), 3) for m in (plain_small, ridge_small)]
```

The scores are 0.585 with almost no penalty and 0.636 for the ridge. With 250 people for 67 columns the unpenalized model sits only 0.085 above a coin flip, and the ridge recovers 0.051 of AUC.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** **Spot the flaw.** A colleague fits the lasso at C = 0.03, sees that it zeroed the two platform columns, and concludes that platform tells us nothing: a model on platform alone would score a coin flip, 0.5. Change the last line: fit a logistic regression on the `platform` column alone on the fitting part, and put its ROC AUC on the validation part in `answer_3`, rounded to 3 decimals.

**Expected:** `answer_3` is a number between 0.5 and 1.

In [ ]:
flat = recipe(0.03, lasso=True).fit(X_fit, y_fit)
zeroed = [n.split("__")[1] for n, c in zip(flat[0].get_feature_names_out(), flat[-1].coef_[0]) if c == 0]
print([n for n in zeroed if n.startswith("platform")])        # the lasso at C = 0.03 gave the platform columns a coefficient of exactly zero
answer_3 = 0.5      # colleague: "the lasso zeroed platform, so platform tells us nothing: its AUC is a coin flip, 0.5"
# change the last line: fit a logistic regression on the platform column alone (fitting part) and give its AUC on the validation part, rounded to 3 decimals

In [ ]:
_ = check("Exercise 3", "answer_3", 0.553, traps={0.5: "That is the colleague's guess. Fit the model on the platform column alone and score it."}, tol=5e-4)

<details>
<summary>Hint</summary>

A coefficient of zero says that the lasso could do without the column at this price, given the other columns. What does a model that has only this column score?

</details>

<details>
<summary>Half-solution</summary>

```python
only_platform = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["platform"])]), LogisticRegression(max_iter=1000)).fit(X_fit, y_fit)
answer_3 = round(float(roc_auc_score(y_val, only_platform.predict_proba(...)[:, 1])), 3)
```

</details>

<details>
<summary>Solution</summary>

```python
only_platform = make_pipeline(ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), ["platform"])]), LogisticRegression(max_iter=1000)).fit(X_fit, y_fit)
answer_3 = round(float(roc_auc_score(y_val, only_platform.predict_proba(X_val)[:, 1])), 3)
```

Platform alone scores 0.553, above a coin flip. The lasso dropped the platform columns because the channel columns, which it kept, already carry most of what platform says: in the dataset's rules the platform mix depends on the channel (the app store brings no web signups, paid social almost none). A zero from the lasso means "not needed, given the others, at this price", not "no signal".

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")